# 02. TPE Contact Detection

**Pipeline step 2 of 3** — classifies candidate particle pairs as *in contact* or *not in contact* using a ResNet18 model.

## Inputs
- Trajectory `.pkl` from Step 1 (positions, radii, boundary flags)
- Photoelastic images (`Ib_*.png`)
- Pre-trained ResNet18 weights (`models/Contact_Detect_ResNet18.pth`)

## Outputs
- `CONTACT_BOND_<EXP_FOLDER>.pkl` — contact bond dataframe with columns:
  `i, j, xi, yi, xj, yj, ri, rj, beta, prob, singular, frame`

## Workflow
1. Build all candidate bonds within $r_i + r_j + d_{tol}$
2. Run ResNet inference to classify each candidate
3. Post-process: drop singular-only pairs, duplicate bulk bonds for symmetry
4. Inspect a sample frame, then save

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import cv2
import sys
from scipy.spatial import distance
import torch
import torch.nn as nn
from torchvision import models as tv_models

### Utility functions

In [ ]:
def get_boundary_pid(f, xmin, xmax, ymin, ymax):
    mask = (
        (f['x'] < xmin) | (f['x'] > xmax) |
        (f['y'] < ymin) | (f['y'] > ymax)
    )
    boundary_pid = f.particle[mask].to_numpy()
    return boundary_pid


def process_singular_bonds(F_out, boundary_pid):
    F_out = F_out.copy()
    boundary_pid = np.array(boundary_pid)
    F_out['singular'] = False

    all_particles = pd.concat([F_out['i'], F_out['j']])
    counts = all_particles.value_counts()

    bulk_particles = counts.index.difference(boundary_pid)
    singular_particles = bulk_particles[counts[bulk_particles] == 1]
    singular_set = set(singular_particles)

    i_singular = F_out['i'].isin(singular_set)
    j_singular = F_out['j'].isin(singular_set)
    F_out = F_out[~(i_singular & j_singular)].reset_index(drop=True)

    # Re-compute masks after row removal
    i_sing = F_out['i'].isin(singular_set)
    j_sing = F_out['j'].isin(singular_set)
    F_out['singular'] = np.where(i_sing, F_out['i'],
                        np.where(j_sing, F_out['j'], -1)).astype(int)
    return F_out


def drop_bulk_duplicate(df, boundary_pid):
    bulk_mask = (~df['i'].isin(boundary_pid)) & (~df['j'].isin(boundary_pid))
    df['pair_key'] = list(zip(df[['i', 'j']].min(axis=1), df[['i', 'j']].max(axis=1)))
    keep_mask = ~(bulk_mask & df.duplicated('pair_key'))
    df = df[keep_mask].reset_index(drop=True)
    return df.drop(columns='pair_key')


def crop_tangent_square(img, center, angle_rad, crop_size):
    """Crop a rotated ROI so the contact lies at the bottom of the image."""
    M = cv2.getRotationMatrix2D(center, np.degrees(angle_rad) - 90, 1.0)
    rotated = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))
    x, y = int(center[0]), int(center[1])
    half = crop_size // 2
    return rotated[y - half:y + half, x - half:x + half]


def duplicate_and_swap_bulk(F_out):
    """Clone bulk rows and swap i↔j so every particle has its contacts listed under i."""
    to_duplicate = F_out[~F_out['j_on_boundary']].copy()
    swap_cols = {
        'i': 'j', 'xi': 'xj', 'yi': 'yj', 'ri': 'rj',
        'j': 'i', 'xj': 'xi', 'yj': 'yi', 'rj': 'ri'
    }
    swap_cols = {k: v for k, v in swap_cols.items() if k in F_out.columns}
    other_cols = [c for c in F_out.columns if c not in swap_cols]
    swapped = to_duplicate.rename(columns=swap_cols)[[*swap_cols.keys(), *other_cols]]
    return pd.concat([F_out, swapped], ignore_index=True)


def get_all_bonds(f, boundary_pid, d_tol):
    """Return all candidate contacts within r_i + r_j + d_tol."""
    coords = f[['x', 'y']].to_numpy()
    radii  = f['rpx'].to_numpy()
    pid    = f['particle'].to_numpy()

    dist_mat = distance.cdist(coords, coords)
    r_mat    = radii[:, None] + radii[None, :]
    nbrs_bool = (dist_mat < r_mat + d_tol) & (dist_mat > 0)
    nbr_i, nbr_j = np.where(nbrs_bool)

    result = np.empty((len(nbr_i), 8), dtype=float)
    result[:, 0]   = pid[nbr_i]
    result[:, 1:3] = coords[nbr_i]
    result[:, 3]   = radii[nbr_i]
    result[:, 4]   = pid[nbr_j]
    result[:, 5:7] = coords[nbr_j]
    result[:, 7]   = radii[nbr_j]

    F_out = pd.DataFrame(result, columns=['i', 'xi', 'yi', 'ri', 'j', 'xj', 'yj', 'rj'])
    F_out[['i', 'j']] = F_out[['i', 'j']].astype(int)
    F_out = F_out[~F_out['i'].isin(set(boundary_pid))].reset_index(drop=True)
    F_out = drop_bulk_duplicate(F_out, boundary_pid)
    F_out['j_on_boundary'] = F_out['j'].isin(boundary_pid)
    F_out = F_out.sort_values(by='i').reset_index(drop=True)
    return F_out, f, nbr_i, nbr_j


# ImageNet normalisation constants (must match training transforms)
_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def predict_contact_batch(f_bond_frame, I, model, plot_raw=False, batch_size=32):
    """
    Crop contact regions, normalise, and run batched ResNet inference.

    Returns:
        preds       – (N, num_classes) float32 softmax probabilities
        batch_crops – (N, 128, 128, 3) uint8 raw crops
    """
    device = next(model.parameters()).device
    n = len(f_bond_frame)
    batch_crops = np.empty((n, 128, 128, 3), dtype=np.uint8)

    for idx, row in enumerate(f_bond_frame.itertuples(index=False)):
        angle    = np.arctan2(row.yj - row.yi, row.xj - row.xi)
        ri       = row.ri
        x1       = int(row.xi + ri * np.cos(angle))
        y1       = int(row.yi + ri * np.sin(angle))
        cropped  = crop_tangent_square(I, (x1, y1), angle, int(1.2 * ri))
        cropped  = cv2.resize(cropped, (128, 128), interpolation=cv2.INTER_AREA)
        cropped  = cv2.cvtColor(cropped, cv2.COLOR_GRAY2RGB)
        batch_crops[idx] = cropped

        if plot_raw:
            plt.figure(figsize=(4, 4))
            plt.imshow(cropped)
            plt.show()

    imgs = torch.from_numpy(batch_crops).float() / 255.0   # (N, H, W, C)
    imgs = imgs.permute(0, 3, 1, 2)                        # (N, C, H, W)
    imgs = (imgs - _IMAGENET_MEAN) / _IMAGENET_STD

    model.eval()
    all_probs = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            logits = model(imgs[start:start + batch_size].to(device))
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())

    return np.concatenate(all_probs, axis=0), batch_crops


def plot_contacts(I, f, F_out, boundary_pid):
    """Render image with particle circles and contact lines. Returns RGB array."""
    fig, ax = plt.subplots(figsize=(16, 9))
    ax.imshow(I, cmap='gray')
    for _, row in f.iterrows():
        ax.add_patch(plt.Circle((row['x'], row['y']), row['rpx'],
                                color='green', fill=False, linewidth=1))
    for _, row in f[f.particle.isin(boundary_pid)].iterrows():
        ax.add_patch(plt.Circle((row['x'], row['y']), row['rpx'],
                                color='red', fill=False, linewidth=1))
    for _, row in F_out.iterrows():
        ax.plot([row['xi'], row['xj']], [row['yi'], row['yj']],
                color='cyan', linestyle='--', linewidth=5, alpha=0.5)
    for _, row in F_out[F_out.singular > 0].iterrows():
        ax.plot([row['xi'], row['xj']], [row['yi'], row['yj']],
                color='magenta', linestyle='--', linewidth=2, alpha=0.4)
    ax.axis('off')
    fig.tight_layout()
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    return img


### Parameters — edit here before running

In [ ]:
EXP_FOLDER = "TPE_20260122A01_N=265x2_5e-4rps_10fps_steady_2000frames"

TRAJ_DIR  = r'M:\Archive\Proj_TPE\Disk_traj_files'   # trajectory .pkl files
IMG_DIR   = r'N:\PROJ_TPE'                            # raw image root
BOND_DIR  = r'M:\Archive\Proj_TPE\Contact_bond_files' # output directory


########### FIXED PARAMETERS (EDIT AS NEEDED) #####################
filetype = ".png"
roi      = (250, 1200, 0, 2000)  # (y_min, y_max, x_min, x_max)
d_tol    = 10                    # neighbour distance tolerance [px]

### Read in tracked file, find all bonds that have length < $r_i + r_j +$ d_tol

output is F_bond that contains all potential bonds and boundary information

1. No boundary particles is on i 
2. Bulk - Bulk bonds are only counted once
3. Bonds with j on boundary are noted

In [ ]:
# Derived paths
linked_file = os.path.join(TRAJ_DIR, f"{EXP_FOLDER}.pkl")
PE_dir      = os.path.join(IMG_DIR, EXP_FOLDER)

F = pd.read_pickle(linked_file)

F_bond = []

# Pre-group to avoid O(N) scan per frame
grouped_F = F.groupby('frame')

for frame in range(1, int(F.frame.max()) + 1):
    sys.stdout.write(f"\rBuilding bonds — frame: {frame}")
    sys.stdout.flush()

    if frame not in grouped_F.groups:
        continue
    f = grouped_F.get_group(frame).reset_index(drop=True)
    boundary_pid = f.particle[f.boundary].to_numpy()

    F_bond_temp, *_ = get_all_bonds(f, boundary_pid, d_tol)
    F_bond_temp['frame'] = frame
    F_bond.append(F_bond_temp)

F_bond = pd.concat(F_bond, ignore_index=True)
print(f"\nDone. {len(F_bond)} candidate bonds across {int(F.frame.max())} frames.")


### Use model to distinguish contact or not, add contact column to F_bond

Cropped square has sides = $1.2 r_i$ 

Each contact point is only examined once

If both particles only have one contact (i.e. singular), the contact is deleted.

If either particle is singular, the contact is kept. The singular id is marked in the "singular" column

If neither are singular, singular column writes -1

Finally, ij pairs that are not on the boundary are cloned and swapped to ji

In [ ]:
NUM_CLASSES = 2  # 0 = no contact, 1 = contact  (must match training)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = os.path.join(os.getcwd(), 'models', 'ResNet18_contact_finetuned.pth')

_backbone = tv_models.resnet18(weights=None)
_backbone.fc = nn.Sequential(
    nn.Linear(_backbone.fc.in_features, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, NUM_CLASSES)
)
_backbone.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model = _backbone.to(device).eval()

print(f"Model loaded: {model_path}")
print(f"Running on:   {device}")

In [ ]:
all_frames = []

# Pre-group both dataframes to avoid O(N) scan per frame
grouped_F    = F.groupby('frame')
grouped_bond = F_bond.groupby('frame')

for frame in range(1, int(F.frame.max()) + 1):
# for frame in [242]:  # single-frame test

    sys.stdout.write(f"\rClassifying contacts — frame: {frame}")
    sys.stdout.flush()

    if frame not in grouped_F.groups or frame not in grouped_bond.groups:
        continue

    # Position data
    f            = grouped_F.get_group(frame).copy()
    boundary_pid = f.particle[f.boundary].to_numpy()

    # Load image
    PE_img_path  = os.path.join(PE_dir, f'Ib_{frame + 1}{filetype}')
    I            = cv2.imread(PE_img_path, cv2.IMREAD_GRAYSCALE)[roi[0]:roi[1], roi[2]:roi[3]]

    # Candidate bonds for this frame
    f_bond_frame = grouped_bond.get_group(frame).copy()

    # Run inference
    preds, _ = predict_contact_batch(f_bond_frame, I, model, plot_raw=False, batch_size=32)

    if preds.ndim != 2:
        raise ValueError(f"Expected (N, 2) predictions, got shape {preds.shape}")

    f_bond_frame['contact'] = np.argmax(preds, axis=1)
    f_bond_frame['prob']    = np.max(preds, axis=1)

    # Keep confirmed contacts only
    f_bond_frame = f_bond_frame[f_bond_frame.contact > 0]

    # Mark / drop singular bonds
    f_bond_frame = process_singular_bonds(f_bond_frame, boundary_pid)

    # Duplicate bulk bonds so every particle has contacts listed under i
    f_bond_frame = duplicate_and_swap_bulk(f_bond_frame)

    all_frames.append(f_bond_frame)

F_contact = pd.concat(all_frames, ignore_index=True).drop(columns=['contact'])
print(f"\nDone. {len(F_contact)} contact bonds across {int(F.frame.max())} frames.")


### Inspect results in one frame

Normal contacts in cyan

Singular contacts in magenta

In [ ]:
for frame in [942]: # this frame is for testing

    PE_img_path = os.path.join(PE_dir, f'Ib_{frame+1}{filetype}')
    I = cv2.imread(PE_img_path, cv2.IMREAD_GRAYSCALE)[roi[0]:roi[1], roi[2]:roi[3]]
    f = F[F.frame==frame].copy() 

    contact = F_contact[F_contact.frame==frame]['i'].value_counts()
    z = contact[contact > 1].mean()
    print(z)

    plt.figure(figsize=(12, 8))
    plt.imshow(plot_contacts(I, f, F_contact[F_contact.frame==frame], f.particle[f.boundary].to_numpy()))
    plt.axis('off')


### Save results to disk

In [ ]:
out_path = os.path.join(BOND_DIR, f"CONTACT_BOND_{EXP_FOLDER}.pkl")
F_contact.to_pickle(out_path)
print(f"Saved: {out_path}")
F_contact